In [ ]:
# ============================
# AUTOENCODER 1D (VERSIONE TESI MIGLIORATA)
# ============================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# ============================
# SEED
# ============================
SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ============================
# PATH
# ============================
out_dir = r"../BLACKBOX_AE1D/"
os.makedirs(out_dir, exist_ok=True)

# ============================
# RISCHIO → NUMERICO (MAX)
# ============================
def trasforma_rischio(df):

    if "rischio" not in df.columns:
        return df

    mapping = {
        "Nessun": 0,
        "Basso": 1,
        "Medio": 2,
        "Alto": 3,
        "Molto alto": 4
    }

    def parse_risk(x):
        if pd.isna(x):
            return np.nan
        
        parts = [p.strip() for p in str(x).split("|")]
        values = [mapping.get(p, np.nan) for p in parts]
        return np.nanmax(values)

    df["rischio_num"] = df["rischio"].apply(parse_risk)
    df = df.drop(columns=["rischio"])

    return df

# ============================
# LOAD
# ============================
def load_data(path):
    df = pd.read_excel(path)
    df["pro_com"] = df["pro_com"].astype(str)
    df = trasforma_rischio(df)
    return df

ambientale = load_data(r"../INDICI 2019/dataset_macroarea_ambientale_2019.xlsx")
socio      = load_data(r"../INDICI 2019/dataset_macroarea_socio_2019.xlsx")
economico  = load_data(r"../INDICI 2019/dataset_macroarea_economica_2019.xlsx")
servizi    = load_data(r"../INDICI 2019/dataset_macroarea_servizi_2019.xlsx")

# ============================
# POLARITÀ
# ============================
direction = {
    "indice_inquinamento": 1,
    "consumo_del_suolo": 1,
    "raccolta_indifferenziata_dei_rifiuti_urbani_per_abitante": 1,
    "tasso_di_motorizzazione_ad_alta_emissione": 1,
    "rischio_num": 1,
    "scarto_aree_protette": -1,

    "ind_vecchiaia_19": 1,
    "quota_pop_attiva_19": -1,
    "Tasso crescita naturale 2019": -1,
    "Tasso migratorio totale 2019": 1,
    "Stranieri/pop 2019": -1,
    "Densità abitativa 2019": -1,

    "Reddito medio per contribuente 2019": -1,
    "imprese_per_1000_ab": -1,
    "laureati_per_1000_ab": -1,
    "Contribuenti IRPEF con reddito complessivo inferiore a 10.000 euro nel 2019 (%)": 1,
    "addetti_100_res": -1,

    "indice_di_accessibilita_ai_servizi_essenziali": 1,
    "sportelli_dev": -1,
    "musei_dev": -1,
    "sanita_dev": -1,
    "Superficie esercizi commerciali per kmq 2019": -1,
    "Addetti dei servizi alla persona per 1.000 abitanti nel 2019": -1
}

# ============================
# PREPROCESS
# ============================
def preprocess(df):

    df = df.copy()

    X = df.select_dtypes(include=[np.number]).copy()

    # polarità
    for col in X.columns:
        if col in direction:
            X[col] *= direction[col]

    # rimuovi NA
    mask = X.notna().all(axis=1)
    X = X.loc[mask]
    ids = df.loc[mask, "pro_com"].reset_index(drop=True).values

    # scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, ids, X.columns

# ============================
# AUTOENCODER 1D
# ============================
def build_autoencoder(input_dim):

    inp = Input(shape=(input_dim,))
    
    x = Dense(input_dim // 2, activation="relu")(inp)
    x = BatchNormalization()(x)
    
    # 🔥 INDICE DIRETTO
    bottleneck = Dense(1, activation="linear", name="index")(x)
    
    x = Dense(input_dim // 2, activation="relu")(bottleneck)
    x = BatchNormalization()(x)
    
    out = Dense(input_dim, activation="linear")(x)

    model = Model(inp, out)
    model.compile(optimizer="adam", loss="mse")

    return model

# ============================
# TRAIN AE
# ============================
def autoencoder_index(X):

    model = build_autoencoder(X.shape[1])

    early_stop = EarlyStopping(
        monitor="loss",
        patience=10,
        restore_best_weights=True
    )

    model.fit(X, X, epochs=200, batch_size=32, verbose=0, callbacks=[early_stop])

    encoder = Model(model.input, model.get_layer("index").output)

    idx = encoder.predict(X).flatten()

    return idx

# ============================
# NORMALIZZAZIONE
# ============================
def normalize_index(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))

# ============================
# RUN
# ============================
datasets = {
    "ambientale": ambientale,
    "socio": socio,
    "economico": economico,
    "servizi": servizi
}

results = []

for name, df in datasets.items():

    print(f"\nRunning AE 1D: {name}")

    X, ids, features = preprocess(df)

    idx = autoencoder_index(X)

    idx = normalize_index(idx)

    # 🔥 orientamento coerente
    if np.corrcoef(idx, X.mean(axis=1))[0,1] < 0:
        idx = 1 - idx

    res = pd.DataFrame({
        "pro_com": ids,
        f"indice_{name}": idx
    })

    results.append(res)

# ============================
# MERGE
# ============================
df_final = results[0]

for df in results[1:]:
    df_final = df_final.merge(df, on="pro_com")

# ============================
# EXPORT
# ============================
df_final.to_excel(
    out_dir + "indici_blackbox_AE1D.xlsx",
    index=False
)

print("\n✅ INDICE BLACKBOX (AE 1D) COMPLETATO")

# ============================
# INTERPRETAZIONE
# ============================
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

def interpret_model(X, idx, feature_names, name):

    print(f"\nInterpretazione: {name}")

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=6,
        random_state=42
    )

    model.fit(X, idx)

    explainer = shap.Explainer(model, X)

    shap_values = explainer(
        X,
        check_additivity=False
    )

    # SUMMARY
    plt.figure()
    shap.summary_plot(
        shap_values,
        X,
        feature_names=feature_names,
        show=False
    )
    plt.title(f"SHAP Summary - {name}")
    plt.savefig(out_dir + f"shap_summary_{name}.png")
    plt.close()

    # BAR
    plt.figure()
    shap.plots.bar(shap_values, show=False)
    plt.title(f"SHAP Importance - {name}")
    plt.savefig(out_dir + f"shap_bar_{name}.png")
    plt.close()

    # CORRELAZIONI
    df_tmp = pd.DataFrame(X, columns=feature_names)
    corr = df_tmp.corrwith(pd.Series(idx))
    corr.to_csv(out_dir + f"corr_{name}.csv")

    # RF importance
    fi = pd.Series(model.feature_importances_, index=feature_names)
    fi.to_csv(out_dir + f"rf_importance_{name}.csv")

    print("✔ Interpretazione completata")

# ============================
# RUN INTERPRETAZIONE
# ============================
for name, df in datasets.items():

    X, ids, features = preprocess(df)
    idx = autoencoder_index(X)
    idx = normalize_index(idx)

    interpret_model(X, idx, features, name)